In [ ]:
# Cell: Fetch + cache histories (run once, then skip)

import os
import pandas as pd
import numpy as np
import wandb
from concurrent.futures import ThreadPoolExecutor, as_completed

WANDB_PROJECT = "aifgen/dm_control_ppo_vs_dart2"
METRIC = "charts/episodic_return"
CACHE_FILE = "dm_control_histories2.parquet"
N_SAMPLES = 2000  # subsample per run
N_WORKERS = 10    # parallel threads

if os.path.exists(CACHE_FILE):
    print(f"Loading cached data from {CACHE_FILE}")
    df = pd.read_parquet(CACHE_FILE)
else:
    print("Fetching run histories from wandb API...")
    api = wandb.Api()
    runs = api.runs(WANDB_PROJECT, per_page=300)
    run_list = list(runs)
    total = len(run_list)
    print(f"Found {total} runs")

    def fetch_one(run):
        env_id = run.config.get("env_id", "")
        exp_name = run.config.get("exp_name", "")
        seed = run.config.get("seed", 0)
        if not env_id or not exp_name:
            return None
        try:
            hist = run.history(keys=[METRIC, "global_step"], samples=N_SAMPLES, pandas=True)
            if hist.empty:
                return None
            hist = hist.dropna(subset=[METRIC, "global_step"])
            if hist.empty:
                return None
            hist["env_id"] = env_id
            hist["exp_name"] = exp_name
            hist["seed"] = seed
            return hist
        except Exception as e:
            print(f"\n  Warning: failed {run.id}: {e}")
            return None

    all_data = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(fetch_one, run): i for i, run in enumerate(run_list)}
        for future in as_completed(futures):
            i = futures[future]
            result = future.result()
            if result is not None:
                all_data.append(result)
            print(f"\r[{len(all_data)} done / {i+1} processed / {total} total]   ", end="", flush=True)

    print(f"\nFetched {len(all_data)} runs")
    df = pd.concat(all_data, ignore_index=True)

print(f"Dataset: {len(df)} rows | {df['env_id'].nunique()} envs | algos: {df['exp_name'].unique().tolist()}")

Plots saved as 'dart_vs_ppo_comparison.png' and 'dart_vs_ppo_comparison.pdf'


/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_13770/2057930297.py:163: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["font.size"] = 46
plt.rcParams["axes.labelsize"] = 58
plt.rcParams["axes.titlesize"] = 66
plt.rcParams["legend.fontsize"] = 46
plt.rcParams["xtick.labelsize"] = 44
plt.rcParams["ytick.labelsize"] = 44

# Create figure (single plot, tall + wide for full-width LaTeX)
fig, ax = plt.subplots(1, 1, figsize=(22, 18), dpi=200)

def exponential_moving_average(data, alpha=0.05):
    """Apply exponential moving average smoothing."""
    ema = np.zeros_like(data)
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

# Define colors
colors = {
    "DART": "#2E86AB",      # Blue
    "PPO": "#A23B72",       # Purple/Pink
}

dart_line_color = "#333333"
dart_line_style = {"color": dart_line_color, "linestyle": "--", "linewidth": 3.5, "alpha": 0.8, "zorder": 5}

# ========================
# LLM PPO vs DART
# ========================
df = pd.read_csv("benchmark_results/LLM_ppo_dart_preliminary.csv")

steps = df["train/episode"].values

dart_mean = df["dart_enabled: true - train/objective/scores"].values
dart_min = df["dart_enabled: true - train/objective/scores__MIN"].values
dart_max = df["dart_enabled: true - train/objective/scores__MAX"].values

ppo_mean = df["dart_enabled: false - train/objective/scores"].values
ppo_min = df["dart_enabled: false - train/objective/scores__MIN"].values
ppo_max = df["dart_enabled: false - train/objective/scores__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean)
steps = steps[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]

dart_ema = exponential_moving_average(dart_mean, alpha=0.05)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.05)

ax.plot(steps, dart_ema, label="DART", color=colors["DART"], linewidth=5)
ax.fill_between(steps, dart_min, dart_max, color=colors["DART"], alpha=0.2)

ax.plot(steps, ppo_ema, label="PPO", color=colors["PPO"], linewidth=5)
ax.fill_between(steps, ppo_min, ppo_max, color=colors["PPO"], alpha=0.2)

ax.axvline(x=20000, **dart_line_style)

ax.set_xlabel("Episode", fontweight="bold")
ax.set_ylabel("Reward Score", fontweight="bold")
ax.legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("dart_vs_ppo_llm.png", dpi=300, bbox_inches="tight")
plt.savefig("dart_vs_ppo_llm.pdf", bbox_inches="tight")
print("Plots saved as 'dart_vs_ppo_llm.png' and 'dart_vs_ppo_llm.pdf'")
plt.show()

Plots saved as 'dart_vs_ppo_llm.png' and 'dart_vs_ppo_llm.pdf'


/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_13770/129839075.py:77: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [6]:
import pandas as pd
import wandb
api = wandb.Api()

# Project is specified by <entity/project-name>
runs = api.runs("aifgen/dm_control_ppo_vs_dart")

summary_list, config_list, name_list = [], [], []
for run in runs:
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)

    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append(
        {k: v for k,v in run.config.items()
          if not k.startswith('_')})

    # .name is the human-readable name of the run.
    name_list.append(run.name)

runs_df = pd.DataFrame({
    "summary": summary_list,
    "config": config_list,
    "name": name_list
    })

runs_df.to_csv("project.csv")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /Users/shahrad/.netrc.


In [2]:
"""
Plot dm_control benchmark results: PPO vs DART vs PPO-Double
Reads wandb export CSV → per-env subplots with EMA-smoothed mean ± std.

Usage:
    pip install wandb pandas matplotlib seaborn
    python plot_benchmark.py
"""

import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
import math

# ─── Config ───
WANDB_PROJECT = "aifgen/dm_control_ppo_vs_dart"
EMA_WEIGHT = 0.6  # wandb default smoothing
METRIC = "charts/episodic_return"
COLS_PER_ROW = 5
ALGO_COLORS = {
    "ppo_dm_control": "#1f77b4",
    "dart_dm_control": "#d62728",
    "ppo_double_dm_control": "#2ca02c",
}
ALGO_LABELS = {
    "ppo_dm_control": "PPO",
    "dart_dm_control": "DART",
    "ppo_double_dm_control": "PPO-Double",
}


def ema_smooth(values, weight=0.6):
    """Exponential moving average, matching wandb's smoothing."""
    smoothed = []
    last = values[0] if len(values) > 0 else 0
    for v in values:
        smoothed_val = last * weight + (1 - weight) * v
        smoothed.append(smoothed_val)
        last = smoothed_val
    return np.array(smoothed)


def fetch_histories(project_path):
    """Download all run histories from wandb API."""
    api = wandb.Api()
    runs = api.runs(project_path, per_page=300)

    all_data = []
    for i, run in enumerate(runs):
        env_id = run.config.get("env_id", "")
        exp_name = run.config.get("exp_name", "")
        seed = run.config.get("seed", 0)

        if not env_id or not exp_name:
            continue

        print(f"\r[{i+1}] Fetching {env_id} / {exp_name} / seed={seed}", end="", flush=True)

        try:
            hist = run.history(keys=[METRIC, "global_step"], samples=500)
            if hist.empty:
                continue
            hist = hist.dropna(subset=[METRIC, "global_step"])
            hist["env_id"] = env_id
            hist["exp_name"] = exp_name
            hist["seed"] = seed
            all_data.append(hist)
        except Exception as e:
            print(f"\n  Warning: failed to fetch {run.id}: {e}")
            continue

    print(f"\nFetched {len(all_data)} runs total")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


def make_plots(df):
    """Generate grid of per-environment learning curves."""
    sns.set_theme(style="whitegrid", font_scale=0.9)

    envs = sorted(df["env_id"].unique())
    n_envs = len(envs)
    n_rows = math.ceil(n_envs / COLS_PER_ROW)

    fig, axes = plt.subplots(
        n_rows, COLS_PER_ROW,
        figsize=(4.5 * COLS_PER_ROW, 3.5 * n_rows),
        squeeze=False,
    )

    for idx, env in enumerate(envs):
        row, col = divmod(idx, COLS_PER_ROW)
        ax = axes[row][col]

        env_df = df[df["env_id"] == env]

        for algo in ["ppo_dm_control", "dart_dm_control", "ppo_double_dm_control"]:
            algo_df = env_df[env_df["exp_name"] == algo]
            if algo_df.empty:
                continue

            # Bin global_step into ~200 bins for alignment across seeds
            step_min = algo_df["global_step"].min()
            step_max = algo_df["global_step"].max()
            bins = np.linspace(step_min, step_max, 200)
            algo_df = algo_df.copy()
            algo_df["step_bin"] = pd.cut(algo_df["global_step"], bins=bins, labels=bins[:-1])
            algo_df["step_bin"] = algo_df["step_bin"].astype(float)

            grouped = algo_df.groupby("step_bin")[METRIC]
            mean = grouped.mean()
            std = grouped.std().fillna(0)

            # Drop NaN bins
            valid = mean.dropna().index
            mean = mean.loc[valid]
            std = std.loc[valid]

            steps = mean.index.values
            mean_vals = ema_smooth(mean.values, EMA_WEIGHT)
            std_vals = ema_smooth(std.values, EMA_WEIGHT)

            color = ALGO_COLORS[algo]
            label = ALGO_LABELS[algo]

            ax.plot(steps, mean_vals, color=color, label=label, linewidth=1.5)
            ax.fill_between(
                steps,
                mean_vals - std_vals,
                mean_vals + std_vals,
                alpha=0.15,
                color=color,
            )

        # Clean env name for title
        short_name = env.replace("dm_control/", "").replace("-v0", "")
        ax.set_title(short_name, fontsize=10, fontweight="bold")
        ax.set_xlabel("Steps", fontsize=8)
        ax.set_ylabel("Return", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(6, 6))

    # Remove empty subplots
    for idx in range(n_envs, n_rows * COLS_PER_ROW):
        row, col = divmod(idx, COLS_PER_ROW)
        fig.delaxes(axes[row][col])

    # Single legend at the top
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="upper center",
        ncol=3,
        fontsize=11,
        frameon=True,
        bbox_to_anchor=(0.5, 1.02),
    )

    fig.suptitle(
        "dm_control Benchmark: PPO vs DART vs PPO-Double\n(10 seeds, 8M steps, EMA smoothed)",
        fontsize=14,
        fontweight="bold",
        y=1.05,
    )

    plt.tight_layout()
    plt.savefig("dm_control_benchmark.png", dpi=200, bbox_inches="tight")
    plt.savefig("dm_control_benchmark.pdf", bbox_inches="tight")
    print("Saved: dm_control_benchmark.png / .pdf")
    plt.show()


if __name__ == "__main__":
    print("Fetching run histories from wandb API...")
    df = fetch_histories(WANDB_PROJECT)

    if df.empty:
        print("ERROR: No data fetched. Check your wandb project path.")
        exit(1)

    print(f"\nDataset: {len(df)} rows")
    print(f"Envs:    {df['env_id'].nunique()}")
    print(f"Algos:   {df['exp_name'].unique().tolist()}")
    print()

    make_plots(df)

Fetching run histories from wandb API...
[932] Fetching dm_control/manipulator-bring_peg-v0 / dart_dm_control / seed=3 seed=100

KeyboardInterrupt: 

In [1]:
# Cell: Fetch + cache histories (run once, then skip)

import os
import pandas as pd
import numpy as np
import wandb
from concurrent.futures import ThreadPoolExecutor, as_completed

WANDB_PROJECT = "aifgen/dm_control_ppo_vs_dart"
METRIC = "charts/episodic_return"
CACHE_FILE = "dm_control_histories.parquet"
N_SAMPLES = 2000  # subsample per run
N_WORKERS = 10    # parallel threads

if os.path.exists(CACHE_FILE):
    print(f"Loading cached data from {CACHE_FILE}")
    df = pd.read_parquet(CACHE_FILE)
else:
    print("Fetching run histories from wandb API...")
    api = wandb.Api()
    runs = api.runs(WANDB_PROJECT, per_page=300)
    run_list = list(runs)
    total = len(run_list)
    print(f"Found {total} runs")

    # Filter out non-finished runs upfront to avoid ValueError on .history()
    valid_runs = [r for r in run_list if r.state in ("finished", "crashed")]
    skipped = total - len(valid_runs)
    print(f"Skipping {skipped} runs (still running/failed) — using {len(valid_runs)} finished/crashed runs")

    def fetch_one(run):
        env_id = run.config.get("env_id", "")
        exp_name = run.config.get("exp_name", "")
        seed = run.config.get("seed", 0)
        if not env_id or not exp_name:
            return None
        try:
            hist = run.history(keys=[METRIC, "global_step"], samples=N_SAMPLES, pandas=True)
            if hist.empty:
                return None
            hist = hist.dropna(subset=[METRIC, "global_step"])
            if hist.empty:
                return None
            hist["env_id"] = env_id
            hist["exp_name"] = exp_name
            hist["seed"] = seed
            return hist
        except (ValueError, Exception) as e:
            print(f"\n  Warning: skipping {run.id} ({run.state}): {e}")
            return None

    all_data = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(fetch_one, run): i for i, run in enumerate(valid_runs)}
        for future in as_completed(futures):
            i = futures[future]
            result = future.result()
            if result is not None:
                all_data.append(result)
            print(f"\r[{len(all_data)} done / {i+1} processed / {len(valid_runs)} total]   ", end="", flush=True)

    print(f"\nFetched {len(all_data)} runs")
    df = pd.concat(all_data, ignore_index=True)

print(f"Dataset: {len(df)} rows | {df['env_id'].nunique()} envs | algos: {df['exp_name'].unique().tolist()}")

Loading cached data from dm_control_histories.parquet
Dataset: 2690000 rows | 45 envs | algos: ['ppo_double_dm_control', 'dart_dm_control', 'ppo_dm_control']


In [10]:
!uv add pyarrow fastparquet
df.to_parquet(CACHE_FILE)
print(f"Cached to {CACHE_FILE}")

Resolved 212 packages in 10ms
Audited 121 packages in 1ms
Cached to dm_control_histories.parquet


In [2]:
df=df[df['exp_name'] != 'ppo_dm_control']

In [5]:
df['exp_name'].unique()

array(['ppo_double_dm_control', 'dart_dm_control'], dtype=object)

In [8]:
# Cell 2: Generate grid plot

import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

EMA_WEIGHT = 0.8
COLS_PER_ROW = 5
METRIC = "charts/episodic_return"

ALGO_ORDER = ["dart_dm_control", "ppo_double_dm_control"]
ALGO_COLORS = {"dart_dm_control": "#2E86AB", "ppo_double_dm_control": "#F18F01"}
ALGO_LABELS = {"dart_dm_control": "DART", "ppo_double_dm_control": "PPO"}


def ema_smooth(values, weight=0.6):
    smoothed = np.zeros_like(values, dtype=float)
    smoothed[0] = values[0]
    for i in range(1, len(values)):
        smoothed[i] = weight * smoothed[i - 1] + (1 - weight) * values[i]
    return smoothed


sns.set_style("whitegrid")
envs = sorted(df["env_id"].unique())
n_envs = len(envs)
n_rows = math.ceil(n_envs / COLS_PER_ROW)

fig, axes = plt.subplots(n_rows, COLS_PER_ROW, figsize=(5 * COLS_PER_ROW, 4 * n_rows), squeeze=False)

for idx, env in enumerate(envs):
    row, col = divmod(idx, COLS_PER_ROW)
    ax = axes[row][col]
    env_df = df[df["env_id"] == env]

    for algo in ALGO_ORDER:
        algo_df = env_df[env_df["exp_name"] == algo]
        if algo_df.empty:
            continue

        step_min, step_max = algo_df["global_step"].min(), algo_df["global_step"].max()
        bins = np.linspace(step_min, step_max, 200)
        algo_df = algo_df.copy()
        algo_df["step_bin"] = pd.cut(algo_df["global_step"], bins=bins, labels=bins[:-1])
        algo_df["step_bin"] = algo_df["step_bin"].astype(float)

        grouped = algo_df.groupby("step_bin")[METRIC]
        mean = grouped.mean().dropna()
        std = grouped.std().fillna(0).loc[mean.index]

        steps = mean.index.values
        mean_vals = ema_smooth(mean.values, EMA_WEIGHT)
        std_vals = ema_smooth(std.values, EMA_WEIGHT)

        color = ALGO_COLORS[algo]
        label = ALGO_LABELS[algo]
        ax.plot(steps, mean_vals, color=color, label=label, linewidth=1.8)
        ax.fill_between(steps, mean_vals - std_vals, mean_vals + std_vals, alpha=0.15, color=color)

    short_name = env.replace("dm_control/", "").replace("-v0", "")
    ax.set_title(short_name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Steps", fontsize=8)
    ax.set_ylabel("Return", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.ticklabel_format(axis="x", style="sci", scilimits=(6, 6))

for idx in range(n_envs, n_rows * COLS_PER_ROW):
    row, col = divmod(idx, COLS_PER_ROW)
    fig.delaxes(axes[row][col])

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=13,
           frameon=True, fancybox=True, shadow=True, bbox_to_anchor=(0.5, 1.01))

fig.suptitle(
    "dm_control Benchmark: PPO vs. DART\n(10 seeds, 8M steps, EMA smoothed)",
    fontsize=16, fontweight="bold", y=1.04,
)

plt.tight_layout()
plt.savefig("dm_control_benchmark.png", dpi=200, bbox_inches="tight")
plt.savefig("dm_control_benchmark.pdf", bbox_inches="tight")
print("Saved: dm_control_benchmark.png / .pdf")
plt.show()

Saved: dm_control_benchmark.png / .pdf


/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_47755/1778625142.py:86: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [12]:
# Cell 3: Final performance table + ranking

import pandas as pd
import numpy as np

METRIC = "charts/episodic_return"
LAST_N = 20  # average over last N logged steps per run

ALGO_LABELS = {"dart_dm_control": "DART", "ppo_double_dm_control": "PPO"}

# For each (env, algo, seed), take mean of last LAST_N steps
final_records = []
for (env, algo, seed), grp in df.groupby(["env_id", "exp_name", "seed"]):
    grp_sorted = grp.sort_values("global_step")
    tail = grp_sorted[METRIC].tail(LAST_N)
    final_records.append({
        "env_id": env,
        "exp_name": algo,
        "seed": seed,
        "final_return": tail.mean(),
    })

final_df = pd.DataFrame(final_records)

# Per (env, algo): mean ± std across seeds
summary = final_df.groupby(["env_id", "exp_name"])["final_return"].agg(["mean", "std", "count"]).reset_index()
summary["label"] = summary["exp_name"].map(ALGO_LABELS)
summary["display"] = summary.apply(lambda r: f"{r['mean']:.1f} ± {r['std']:.1f}", axis=1)

# Pivot to table
table = summary.pivot(index="env_id", columns="label", values="display")
table.index = table.index.str.replace("dm_control/", "").str.replace("-v0", "")
table = table[["DART", "PPO"]]  # fix column order
table.index.name = "Environment"

print("=" * 80)
print("Final Episodic Return (mean ± std over seeds, averaged over last 20 steps)")
print("=" * 80)
print(table.to_string())
print()

# Bold the best per env
mean_pivot = summary.pivot(index="env_id", columns="label", values="mean")
mean_pivot.index = mean_pivot.index.str.replace("dm_control/", "").str.replace("-v0", "")
mean_pivot = mean_pivot[["DART", "PPO"]]

best_per_env = mean_pivot.idxmax(axis=1)
print("Best algorithm per environment:")
print(best_per_env.to_string())
print()

# Win counts
win_counts = best_per_env.value_counts()
print("Win counts:")
print(win_counts.to_string())
print()

# Overall ranking: mean of per-env means
overall = mean_pivot.mean(axis=0).sort_values(ascending=False)
print("Overall ranking (mean of per-env mean final return):")
for rank, (algo, val) in enumerate(overall.items(), 1):
    print(f"  #{rank} {algo}: {val:.1f}")

# Also save as LaTeX
# latex = table.to_latex(caption="Final episodic return (mean $\\pm$ std, last 20 steps, 10 seeds)")
# with open("dm_control_final_table.tex", "w") as f:
#     f.write(latex)
# print("\nSaved LaTeX table to dm_control_final_table.tex")

Final Episodic Return (mean ± std over seeds, averaged over last 20 steps)
label                             DART            PPO
Environment                                          
acrobot-swingup             28.9 ± 8.0    24.9 ± 11.3
acrobot-swingup_sparse       2.1 ± 1.5      1.7 ± 1.5
ball_in_cup-catch         843.6 ± 38.8   855.8 ± 39.7
cartpole-balance          758.6 ± 31.1   746.2 ± 27.6
cartpole-balance_sparse   968.8 ± 21.5   956.7 ± 33.1
cartpole-swingup          602.8 ± 26.0   600.1 ± 25.3
cartpole-swingup_sparse  357.9 ± 203.2  230.9 ± 258.1
cartpole-three_poles       148.7 ± 4.9    149.1 ± 4.9
cartpole-two_poles         201.4 ± 6.1    209.6 ± 8.3
cheetah-run               309.0 ± 29.3    71.8 ± 18.4
dog-fetch                    8.8 ± 1.9      9.3 ± 1.8
dog-run                     14.4 ± 3.3     16.9 ± 5.0
dog-stand                  59.2 ± 12.0    83.6 ± 20.2
dog-trot                    23.6 ± 4.2     23.2 ± 4.9
dog-walk                    30.6 ± 5.7     33.7 ± 4.0
finger-